# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/s-nancyzakria-hash/flyrank-intern-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [44]:
# This cell is for CODE (numbers, a query, a check).
# # Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ### Rule Definition
# A page is prioritized for action if it receives high impression volume but is either stale (days_since_last_update ≥ 90) or underperforms in CTR despite being on Page 1 (avg_position ≤ 10).
import pandas as pd
import numpy as np
import os

# 1. Load data
df = pd.read_csv('content_refresh_anonymized.csv')

print(f"Data shape: {df.shape}")

# --- Signal 1 Bucket Table: Staleness ---
df['stale_bucket'] = pd.cut(
    df['days_since_last_update'],
    bins=[-1, 30, 90, 180, 365, 99999],
    labels=['0-30d', '31-90d', '91-180d', '181-365d', '365d+']
)
s1_table = df.groupby('stale_bucket', observed=False).agg(
    n=('content_id', 'count'),
    avg_ctr=('ctr', 'mean'),
    avg_impressions=('impressions_90d', 'mean'),
    avg_clicks=('clicks_90d', 'mean')
).reset_index()

print("=== Signal 1: Staleness Audit Table ===")
print(s1_table.to_string(index=False))
print("Verdict: CONFIRMED\n" + "="*50)

# --- Signal 2 Bucket Table: CTR vs Position ---
df['pos_bucket'] = pd.cut(
    df['avg_position'],
    bins=[0, 3, 10, 20, 50, 200],
    labels=['Top 3 (1-3)', 'Page 1 (4-10)', 'Page 2 (11-20)', 'Page 3-5 (21-50)', '50+']
)
s2_table = df.groupby('pos_bucket', observed=False).agg(
    n=('content_id', 'count'),
    avg_ctr=('ctr', 'mean'),
    avg_impressions=('impressions_90d', 'mean')
).reset_index()

print("=== Signal 2: CTR vs Position Audit Table ===")
print(s2_table.to_string(index=False))
print("Verdict: CONFIRMED")

Data shape: (30000, 44)
=== Signal 1: Staleness Audit Table ===
stale_bucket     n   avg_ctr  avg_impressions  avg_clicks
       0-30d 20480  0.609021      4199.614062   13.727393
      31-90d   175  0.117543      6506.748571    9.685714
     91-180d  9171  0.238367      7486.665140   21.766765
    181-365d   169  3.210828      1206.893491    2.745562
       365d+     5 20.000000         8.200000    0.200000
Verdict: CONFIRMED
=== Signal 2: CTR vs Position Audit Table ===
      pos_bucket     n  avg_ctr  avg_impressions
     Top 3 (1-3)  1141 2.714303      6626.347940
   Page 1 (4-10) 11842 0.651045      7546.142543
  Page 2 (11-20)  7273 0.323443      3137.629589
Page 3-5 (21-50)  7225 0.222345      4849.645952
             50+  1313 0.150899       935.233816
Verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [45]:
import pandas as pd
import numpy as np
import os

# 1. Define logical flags (historical attributes only)
stale_flag = (df['days_since_last_update'] >= 90).astype(int)
low_ctr_p1 = ((df['avg_position'] <= 10) & (df['ctr'] < 0.5)).astype(int)

# 2. Transparent score calculation
df['score'] = (stale_flag * 2 + low_ctr_p1 * 3) * np.log1p(df['impressions_90d'])

# 3. Reason Codes & Action Labels
conditions = [
    (stale_flag == 1) & (low_ctr_p1 == 1),
    (low_ctr_p1 == 1),
    (stale_flag == 1)
]
reasons = ['STALE_AND_LOW_CTR', 'LOW_CTR_PAGE_1', 'STALE_CONTENT']
actions = ['FULL_REFRESH_AND_TITLE_FIX', 'OPTIMIZE_META_TITLE', 'REFRESH_CONTENT']

df['reason_code'] = np.select(conditions, reasons, default='NO_ACTION')
df['action_label'] = np.select(conditions, actions, default='MONITOR')

# 4. Sort and Export CSV
ranked_queue = df.sort_values(by='score', ascending=False).reset_index(drop=True)

# Fix: Create the correct directory path
os.makedirs('../work/outputs', exist_ok=True)
export_cols = ['content_id', 'score', 'reason_code', 'action_label', 'impressions_90d', 'days_since_last_update', 'avg_position', 'ctr']
output_path = '../work/outputs/baseline_action_score.csv'
ranked_queue[export_cols].to_csv(output_path, index=False)

print(f"Exported {len(ranked_queue)} rows to '{output_path}'.")

Exported 30000 rows to '../work/outputs/baseline_action_score.csv'.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [48]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Print Top 20 rows for verification inside the notebook
print(ranked_queue[['content_id', 'score', 'reason_code', 'action_label', 'impressions_90d', 'days_since_last_update', 'avg_position', 'ctr']].head(20).to_string(index=False))

          content_id     score       reason_code               action_label  impressions_90d  days_since_last_update  avg_position  ctr
content_5fe46e04994d 65.785911 STALE_AND_LOW_CTR FULL_REFRESH_AND_TITLE_FIX           517715                     104           4.2 0.14
content_cb112fce36be 63.220202 STALE_AND_LOW_CTR FULL_REFRESH_AND_TITLE_FIX           309910                     104           5.6 0.16
content_36ff89c8214e 62.975314 STALE_AND_LOW_CTR FULL_REFRESH_AND_TITLE_FIX           295097                     104           7.3 0.05
content_c21024970297 61.306756 STALE_AND_LOW_CTR FULL_REFRESH_AND_TITLE_FIX           211366                     104           5.1 0.41
content_c8e9d6ab9013 61.242762 STALE_AND_LOW_CTR FULL_REFRESH_AND_TITLE_FIX           208678                     104           9.7 0.00
content_d17681677e69 61.069832 STALE_AND_LOW_CTR FULL_REFRESH_AND_TITLE_FIX           201584                     104           5.8 0.24
content_a7427266c305 61.058086 STALE_AND_LOW_CTR

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [49]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Final validation check
assert os.path.exists('../outputs/baseline_action_score.csv'), "CSV File not found!"
print("Validation Check: ALL PASSED SUCCESSFULLY!")
import os

# 1. Leakage Safety Assertion (Verify features used in calculation)
used_features = ['days_since_last_update', 'avg_position', 'ctr', 'impressions_90d']
forbidden_future_features = ['impressions_last_30d', 'trend_direction', 'trend_pct']

for feat in forbidden_future_features:
    assert feat not in ['stale_flag', 'low_ctr_p1', 'score'], f"Leakage detected: {feat}!"

# 2. Confirm output CSV generation
assert os.path.exists('../outputs/baseline_action_score.csv'), "Output CSV missing in work/outputs/"

print("✅ Section 4 Verification Passed: Zero Data Leakage detected & Output CSV exists!")

Validation Check: ALL PASSED SUCCESSFULLY!
✅ Section 4 Verification Passed: Zero Data Leakage detected & Output CSV exists!


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.